# Exploring the data products of the dbv-pipeline data run

A template for interactive investigation of what `pipelines/data_dp2.yaml` writes:

| Product | Stage | Section |
|---|---|---|
| `selection_catalog.pq` | SelectSources | 2 |
| `isolated_cutouts/` | MakeCutouts | 3, 4 |
| `psf_stats.yml` | MeasurePSF | 5 |
| `empty_cutouts/`, `noise_model.yml` | MakeCutouts, MeasureNoise | 6 |
| `qa_*/` figures | every stage | 7 |
| the coadd around an object (Butler) | - | 8 (optional, `lsst-weekly` kernel only) |

**How to use it.** Copy this notebook (keep the template clean), set the paths in the first code cell and run top to bottom. Every section checks that its product exists and is skipped otherwise, so it also works on a partly finished run. The figures reuse `src/dbv_pipeline/qa.py`, so they match the pipeline's QA. Section 9 is an empty scratch area.

**Kernel.** `lsst-weekly` (needed for section 8) or `debvader`. The package `dbv_pipeline` is *not* imported (it needs ceci); `qa.py` is loaded directly from its file.

**Memory.** Datasets are opened memory-mapped: nothing is read until it is indexed. Section 6 reads the whole empty-sky set (about 70 MB per 400 stamps).

## 1. Paths and set-up

In [ ]:
from pathlib import Path

# ---- edit these --------------------------------------------------------------------------------------------
REPO = Path("/global/cfs/cdirs/m1727/laliotis/debvader_pipeline/dbv-pipeline")
RUN = Path("/pscratch/sd/l/laliotis/dbv/test1")          # one test / run
DATA = RUN / "data_dp2"                                  # output_dir of pipelines/data_dp2.yaml
WORK = RUN / "work" / "dp2_cutouts"                      # DP2-cutouts working folder (global.work_dir / dp2_cutouts)

SELECTION_CATALOG = DATA / "selection_catalog.pq"
ISOLATED = DATA / "isolated_cutouts"
EMPTY = DATA / "empty_cutouts"
PSF_STATS = DATA / "psf_stats.yml"
NOISE_MODEL = DATA / "noise_model.yml"
CONFIG = REPO / "pipelines" / "config_test1.yaml"       # for the band list and pixel scale if a product lacks them
SEED = 0                                                 # for the random picks
# ------------------------------------------------------------------------------------------------------------

for name, p in [("selection catalogue", SELECTION_CATALOG), ("isolated cutouts", ISOLATED), ("empty cutouts", EMPTY),
                ("psf_stats", PSF_STATS), ("noise_model", NOISE_MODEL), ("config", CONFIG)]:
    print(f"{'found  ' if p.exists() else 'MISSING'}  {name:20s} {p}")

In [ ]:
import importlib.util
import json

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from IPython.display import Image, display

%matplotlib inline

# qa.py straight from the file (importing the dbv_pipeline package would need ceci)
_spec = importlib.util.spec_from_file_location("dbv_qa", REPO / "src" / "dbv_pipeline" / "qa.py")
qa = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(qa)
qa.style()

cfg = yaml.safe_load(open(CONFIG))["global"] if CONFIG.exists() else {}
BANDS = list(cfg.get("bands", list("ugrizy")))
PIXEL_SCALE = float(cfg.get("pixel_scale", 0.2))
rng = np.random.default_rng(SEED)

In [ ]:
# ---- helpers -----------------------------------------------------------------------------------------------
def load_dataset(d):
    # A DebvaderDataset directory -> dict: memory-mapped arrays, `meta` as a DataFrame, dataset.json, stats.json.
    d = Path(d)
    ds = {"dir": d}
    for name in ("images", "variance", "mask", "psf", "targets"):
        if (d / f"{name}.npy").exists():
            ds[name] = np.load(d / f"{name}.npy", mmap_mode="r")
    ds["info"] = json.load(open(d / "dataset.json")) if (d / "dataset.json").exists() else {}
    ds["stats"] = json.load(open(d / "stats.json")) if (d / "stats.json").exists() else {}
    ds["bands"] = list(ds["info"].get("bands") or BANDS)
    ds["meta"] = meta_frame(d / "meta.npz", ds["bands"]) if (d / "meta.npz").exists() else pd.DataFrame()
    return ds


def meta_frame(path, bands):
    # meta.npz -> one row per object. (N, C) arrays become one column per band, e.g. psf_fwhm_i.
    m = np.load(path)
    n = len(m["split"])
    cols = {}
    for k in m.files:
        a = m[k]
        if a.ndim == 1 and len(a) == n:
            cols[k] = a
        elif a.ndim == 2 and a.shape == (n, len(bands)):
            for c, b in enumerate(bands):
                cols[f"{k}_{b}"] = a[:, c]
    df = pd.DataFrame(cols)
    df["split_name"] = df["split"].map({0: "train", 1: "val", 2: "test"})
    return df


def row_of(ds, object_id):
    # Row index in a dataset of a Rubin objectId.
    hit = np.flatnonzero(ds["meta"]["metadata_id"].to_numpy() == object_id)
    if len(hit) == 0:
        raise KeyError(f"{object_id} is not in {ds['dir']}")
    return int(hit[0])


def sym_limits(a, q=99.5):
    v = np.nanpercentile(np.abs(a), q)
    return -v, v


def show_object(ds, k, bands=None, title=None):
    # Image, variance, mask and PSF of row k, one column per band.
    bands = bands or ds["bands"]
    cols = [ds["bands"].index(b) for b in bands]
    rows = [("image", "images"), ("variance", "variance"), ("mask (True = ignore)", "mask"), ("PSF (log10)", "psf")]
    rows = [r for r in rows if r[1] in ds]
    fig, axes = plt.subplots(len(rows), len(cols), figsize=(1.9 * len(cols), 1.95 * len(rows)), squeeze=False)
    for r, (label, key) in enumerate(rows):
        for j, c in enumerate(cols):
            ax = axes[r, j]
            a = np.asarray(ds[key][k][..., c], float)
            if key == "images":
                lo, hi = sym_limits(a)
                ax.imshow(a, origin="lower", cmap="RdBu_r", vmin=lo, vmax=hi)
            elif key == "psf":
                ax.imshow(np.log10(np.clip(a, a.max() * 1e-4, None)), origin="lower", cmap="magma")
            elif key == "mask":
                ax.imshow(a, origin="lower", cmap="gray_r", vmin=0, vmax=1)
            else:
                ax.imshow(a, origin="lower", cmap="viridis")
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
            if r == 0:
                ax.set_title(bands[j], loc="center")
        axes[r, 0].set_ylabel(label, fontsize=8)
    if title is None and len(ds["meta"]):
        m = ds["meta"].iloc[k]
        title = f"row {k}"
        for col, fmt in (("metadata_id", "id {}"), ("metadata_magnitude", "i = {:.2f}"), ("split_name", "{}"),
                         ("masked_fraction", "masked {:.2f}")):
            if col in m:
                title += "   " + fmt.format(m[col])
    fig.suptitle(title or f"row {k}", x=0.01, ha="left", fontsize=10)
    fig.tight_layout()
    return fig


def show_many(ds, rows, band="i", ncols=8, key="images", label=None):
    # One band of several rows, each with its own symmetric stretch, titled by `label(row)` if given.
    c = ds["bands"].index(band)
    nrows = int(np.ceil(len(rows) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(1.6 * ncols, 1.75 * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.set_axis_off()
    for ax, k in zip(axes.ravel(), rows):
        a = np.asarray(ds[key][k][..., c], float)
        lo, hi = sym_limits(a)
        ax.imshow(a, origin="lower", cmap="RdBu_r" if key == "images" else "magma", vmin=lo if key == "images" else None,
                  vmax=hi if key == "images" else None)
        ax.set_title(label(k) if label else str(k), fontsize=7, loc="center")
    fig.tight_layout()
    return fig


def show_yaml(path, keys=None):
    d = yaml.safe_load(open(path))
    d.pop("provenance", None)
    print(yaml.safe_dump({k: d[k] for k in keys} if keys else d, sort_keys=False))
    return d

## 2. Selection catalogue (SelectSources)

One row per Rubin galaxy that copaings kept, with each catalogue cut as a value and a pass flag, and `selected` = cut 1 & cut 2 & cut 6. `cat` is a plain DataFrame: filter it with `cat.query(...)` or boolean masks.

In [ ]:
cat = pd.read_parquet(SELECTION_CATALOG) if SELECTION_CATALOG.exists() else None
if cat is not None:
    print(f"{len(cat):,} objects, {int(cat['selected'].sum()):,} selected, {cat.shape[1]} columns")
    flags = [c for c in cat.columns if c.endswith("_pass")] + ["selected"]
    display(cat[flags].astype("boolean").sum().rename("pass").to_frame().T)
    display(cat.groupby(["tract", "patch"])["selected"].agg(["size", "sum"]).rename(columns={"size": "objects", "sum": "selected"}))

In [ ]:
# All columns, grouped by prefix, to know what can be plotted.
if cat is not None:
    groups = {}
    for c in cat.columns:
        groups.setdefault(c.split("_")[0], []).append(c)
    for g, cs in sorted(groups.items()):
        print(f"{g:12s} {', '.join(cs)}")

In [ ]:
# Magnitude, isolation and sky position: selected against the parent sample (galaxies passing the S/N cut).
if cat is not None:
    sel = cat["selected"].to_numpy(bool)
    parent = cat["cut6_pass"].to_numpy(bool)
    mag = cat["i_cModel_mag"].to_numpy(float)
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), gridspec_kw={"wspace": 0.3})
    bins = np.arange(17, 28.01, 0.25)
    qa.step_hist(axes[0], mag[parent], bins, qa.NEUTRAL, f"S/N cut ({parent.sum():,})")
    qa.step_hist(axes[0], mag[sel], bins, qa.SERIES[0], f"selected ({sel.sum():,})", fill=True)
    axes[0].set_xlabel("i cModel mag"); axes[0].legend(); axes[0].set_title("Magnitude")
    sb = cat["sb_euclid"].to_numpy(float)
    qa.step_hist(axes[1], sb[parent & (sb >= 0)], np.linspace(0, 2, 41), qa.NEUTRAL, "S/N cut, S_b >= 0")
    axes[1].set_yscale("log"); axes[1].set_xlabel("blending entropy S_b"); axes[1].set_title("Blending entropy")
    axes[2].scatter(cat["ra"][parent & ~sel], cat["dec"][parent & ~sel], s=1, color=qa.NEUTRAL, lw=0, label="not selected")
    axes[2].scatter(cat["ra"][sel], cat["dec"][sel], s=2, color=qa.SERIES[0], lw=0, label="selected")
    axes[2].invert_xaxis(); axes[2].set_xlabel("RA"); axes[2].set_ylabel("Dec"); axes[2].legend(markerscale=5)
    axes[2].set_title("Sky")

In [ ]:
# Template for your own question: e.g. how the selected fraction depends on a column, in bins.
if cat is not None:
    COLUMN, BINS = "i_cModel_mag", np.arange(18, 27.01, 0.5)      # <- change
    base = cat[cat["cut6_pass"]]
    x, f, err, n = qa.binned_fraction(base[COLUMN].to_numpy(float), base["selected"].to_numpy(bool), BINS)
    fig, ax = plt.subplots(figsize=(6, 3.4))
    ax.errorbar(x, f, err, marker="o", color=qa.SERIES[0])
    ax.set_xlabel(COLUMN); ax.set_ylabel("selected fraction"); ax.set_ylim(0, 1)

## 3. Isolated cutouts (MakeCutouts)

`iso["images"]`, `iso["variance"]`, `iso["mask"]` are `(N, H, W, C)`; `iso["psf"]` is `(N, P, P, C)`; `iso["meta"]` has one row per stamp (row `k` of the arrays). `iso_cat` joins it with the selection catalogue on the Rubin objectId, so that every catalogue column is available per stamp.

In [ ]:
iso = load_dataset(ISOLATED) if (ISOLATED / "images.npy").exists() else None
if iso is not None:
    print(json.dumps({k: v for k, v in iso["info"].items() if k != "source"}, indent=1))
    print("arrays:", {k: v.shape for k, v in iso.items() if isinstance(v, np.ndarray)})
    print("scale per band (stats.json):", dict(zip(iso["bands"], np.round(iso["stats"]["nonlinear"]["scale_per_band"], 3))))
    display(iso["meta"]["split_name"].value_counts().to_frame().T)
    iso_cat = iso["meta"]
    if cat is not None and "metadata_id" in iso["meta"]:
        iso_cat = iso["meta"].merge(cat, left_on="metadata_id", right_on="object_id", how="left", suffixes=("", "_cat"))
        print(f"joined with the selection catalogue: {iso_cat['object_id'].notna().sum():,} / {len(iso_cat):,} matched")

In [ ]:
# Montage of random stamps (the same figure as qa_cutouts).
if iso is not None:
    fig = qa.make_cutout_qa(ISOLATED, iso["bands"], PIXEL_SCALE, n=24, seed=SEED)

In [ ]:
# One object in every band: image, variance, mask, PSF. Pick a row, or an objectId with row_of(iso, OBJECT_ID).
if iso is not None:
    K = int(rng.integers(len(iso["images"])))          # <- or e.g. row_of(iso, 1234567890)
    fig = show_object(iso, K)
    display(iso_cat.iloc[[K]].T.head(40))

In [ ]:
# Sorted views: the brightest, faintest, most masked ... (change the column and the band).
if iso is not None:
    COLUMN, BAND, N_SHOW = "masked_fraction", "i", 16
    if COLUMN in iso_cat:
        order = np.argsort(iso_cat[COLUMN].to_numpy(float))[::-1][:N_SHOW]
        fig = show_many(iso, order, band=BAND, label=lambda k: f"{COLUMN} {iso_cat[COLUMN].iloc[k]:.3g}")

In [ ]:
# Masks and pixel statistics: masked fraction (companion pixels filled with sky), bad-pixel fraction per band.
if iso is not None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
    if "masked_fraction" in iso_cat:
        qa.step_hist(axes[0], iso_cat["masked_fraction"], np.linspace(0, 1, 41), qa.SERIES[0], None, fill=True)
        axes[0].set_xlabel("fraction of the stamp filled with empty sky"); axes[0].set_title("Companion masking")
    take = np.sort(rng.choice(len(iso["mask"]), size=min(2000, len(iso["mask"])), replace=False))
    bad = np.asarray(iso["mask"][take]).mean(axis=(1, 2))            # (n, C)
    for c, b in enumerate(iso["bands"]):
        qa.step_hist(axes[1], bad[:, c], np.linspace(0, max(0.05, bad.max()), 41), qa.SERIES[c], b, lw=1.5)
    axes[1].set_yscale("log"); axes[1].set_xlabel("fraction of pixels in mask.npy"); axes[1].legend(ncols=2)
    axes[1].set_title(f"Bad pixels ({len(take)} stamps)")

## 4. Per-object properties of the isolated sample

Free-form scatter of any two columns of `iso_cat` (meta.npz + selection catalogue), coloured by a third.

In [ ]:
if iso is not None:
    X, Y, COLOUR = "metadata_magnitude", "psf_fwhm_i", "masked_fraction"      # <- any columns of iso_cat
    have = [c for c in (X, Y, COLOUR) if c in iso_cat]
    if len(have) == 3:
        fig, ax = plt.subplots(figsize=(6.4, 4.4))
        s = ax.scatter(iso_cat[X], iso_cat[Y], c=iso_cat[COLOUR], s=5, lw=0, cmap="viridis")
        fig.colorbar(s, label=COLOUR); ax.set_xlabel(X); ax.set_ylabel(Y)
    else:
        print("missing columns:", sorted({X, Y, COLOUR} - set(have)))

## 5. PSF (MeasurePSF)

`psf_stats.yml` holds the FWHM percentiles per band, the configured internal PSF `G_int`, the suggested FWHM and `ok`. The FWHM per object is `meta.npz["psf_fwhm"]` (`psf_fwhm_<band>` in `iso_cat`). The coadd PSF is constant over 150 px cells, so its map on the sky shows the cell pattern.

In [ ]:
psf_stats = show_yaml(PSF_STATS) if PSF_STATS.exists() else None

In [ ]:
if iso is not None and "psf_fwhm_i" in iso_cat:
    fw = np.stack([iso_cat[f"psf_fwhm_{b}"].to_numpy(float) for b in iso["bands"]], axis=-1)
    checks = None
    if psf_stats is not None:      # rebuild what qa.make_psf_qa needs from psf_stats.yml
        g = psf_stats["internal_psf"]["fwhm_arcsec"]
        g = g if isinstance(g, list) else [g] * len(iso["bands"])
        checks = {b: {"fwhm": g[c], "suggested_fwhm": psf_stats["suggested_internal_psf_fwhm"][b],
                      "ok": psf_stats["ok_per_band"][b]} for c, b in enumerate(iso["bands"])}
    if checks:
        fig, _ = qa.make_psf_qa(fw, np.zeros_like(fw), checks, iso["bands"])   # ellipticity panel: see qa_psf
        plt.close(_)

In [ ]:
# FWHM on the sky, one panel per band (tract pixels if exported, else RA / Dec).
if iso is not None and "psf_fwhm_i" in iso_cat:
    xk, yk = ("x_tract", "y_tract") if "x_tract" in iso_cat else ("ra", "dec")
    fig, axes = plt.subplots(2, 3, figsize=(13, 7.4), squeeze=False)
    for ax, b in zip(axes.ravel(), iso["bands"]):
        s = ax.scatter(iso_cat[xk], iso_cat[yk], c=iso_cat[f"psf_fwhm_{b}"], s=4, lw=0, cmap="viridis")
        fig.colorbar(s, ax=ax, label="FWHM [arcsec]"); ax.set_title(b); ax.set_aspect("equal")
    fig.tight_layout()

In [ ]:
# Mean PSF per band and its radial profile; the widest and narrowest PSF stamps in one band.
if iso is not None:
    take = np.sort(rng.choice(len(iso["psf"]), size=min(2000, len(iso["psf"])), replace=False))
    mean_psf = np.asarray(iso["psf"][take], float).mean(axis=0)                      # (P, P, C)
    P = mean_psf.shape[0]
    yy, xx = np.mgrid[:P, :P] - P // 2
    r = np.hypot(xx, yy).astype(int)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    for c, b in enumerate(iso["bands"]):
        prof = np.bincount(r.ravel(), mean_psf[..., c].ravel()) / np.bincount(r.ravel())
        axes[0].plot(np.arange(len(prof)) * PIXEL_SCALE, prof / prof[0], color=qa.SERIES[c], label=b, lw=1.5)
    axes[0].set_yscale("log"); axes[0].set_ylim(1e-5, 1.2); axes[0].set_xlim(0, P // 2 * PIXEL_SCALE)
    axes[0].set_xlabel("radius [arcsec]"); axes[0].set_ylabel("mean PSF / peak"); axes[0].legend(ncols=2)
    axes[0].set_title("Mean PSF radial profile")
    axes[1].imshow(np.log10(np.clip(mean_psf[..., iso["bands"].index("i")], 1e-6, None)), origin="lower", cmap="magma")
    axes[1].set_title("mean i-band PSF (log10)"); axes[1].grid(False)
    if "psf_fwhm_i" in iso_cat:
        order = np.argsort(iso_cat["psf_fwhm_i"].to_numpy(float))
        pick = np.r_[order[:8], order[-8:]]
        fig2 = show_many(iso, pick, band="i", key="psf", label=lambda k: f"FWHM {iso_cat['psf_fwhm_i'].iloc[k]:.2f}\"")
        fig2.suptitle("i-band PSF: 8 narrowest (first row), 8 widest (second row)", x=0.01, ha="left", fontsize=10)

## 6. Empty sky and the noise model (MeasureNoise)

The whitened pixels `image / sqrt(variance)` of the empty-sky stamps should be unit Gaussian after dividing by the noise factor `f_b`. The per-stamp chi2 / pixel on the sky shows whether some regions are noisier than their variance plane says (undetected sources, bad calibration).

In [ ]:
noise = show_yaml(NOISE_MODEL) if NOISE_MODEL.exists() else None
emp = load_dataset(EMPTY) if (EMPTY / "images.npy").exists() else None
if emp is not None:
    print({k: v.shape for k, v in emp.items() if isinstance(v, np.ndarray)})
    display(emp["meta"]["split_name"].value_counts().to_frame().T)

In [ ]:
if emp is not None:
    images, var, mask = (np.asarray(emp[k]) for k in ("images", "variance", "mask"))
    bad = mask | ~(np.isfinite(var) & (var > 0))
    with np.errstate(invalid="ignore", divide="ignore"):
        white = np.where(bad, np.nan, images / np.sqrt(var))
    if noise is not None:
        f_b, f_mean, f_mad = (np.asarray(noise[k], float) for k in ("noise_factor", "noise_factor_mean", "noise_factor_mad"))
    else:                                         # no noise model yet: compute the two estimators here
        flat = white.reshape(-1, white.shape[-1])
        f_mean = np.nanmean(flat**2, axis=0)
        f_mad = (1.4826 * np.nanmedian(np.abs(flat - np.nanmedian(flat, axis=0)), axis=0))**2
        f_b = f_mad
    figs = qa.make_noise_qa(white, f_b, f_mean, f_mad, emp["bands"])

In [ ]:
# chi2 / pixel of each empty stamp on the sky (after f_b), and the empty-sky montage.
if emp is not None:
    with np.errstate(invalid="ignore"):
        chi2 = np.nanmean(white**2, axis=(1, 2)) / f_b                                    # (N, C)
    xk, yk = ("x_tract", "y_tract") if "x_tract" in emp["meta"] else ("ra", "dec")
    fig, axes = plt.subplots(2, 3, figsize=(13, 7.4), squeeze=False)
    for c, (ax, b) in enumerate(zip(axes.ravel(), emp["bands"])):
        s = ax.scatter(emp["meta"][xk], emp["meta"][yk], c=chi2[:, c], s=10, lw=0, cmap="RdBu_r", vmin=0.8, vmax=1.2)
        fig.colorbar(s, ax=ax, label="chi2 / pixel"); ax.set_title(b); ax.set_aspect("equal")
    fig.tight_layout()
    fig = qa.make_cutout_qa(EMPTY, emp["bands"], PIXEL_SCALE, n=24, seed=SEED, empty=True)

## 7. The pipeline's own QA figures

In [ ]:
for d in sorted(DATA.glob("qa_*")) if DATA.exists() else []:
    pngs = sorted(d.glob("*.png"))
    print(f"\n{d.name}: {len(pngs)} figures")
    for p in pngs:                      # comment out to only list them
        print(p.name)
        display(Image(filename=str(p), width=900))

## 8. Optional: the coadd around an object (Butler, `lsst-weekly` kernel)

A box larger than the stamp, read with the Butler, with the stamp outline drawn. Useful to see the companions that were masked and filled with sky.

In [ ]:
FACTOR = 3          # box size in units of the stamp
BAND = "i"
try:
    import lsst.geom as geom
    from lsst.daf.butler import Butler
    from lsst.images import Box
    have_butler = True
except ImportError:
    have_butler = False
    print("No LSST stack in this kernel: section skipped.")

if have_butler and iso is not None and {"ra", "dec", "tract", "patch"} <= set(iso_cat.columns):
    b = cfg["dp2_field"]["butler"]
    butler = Butler(b["repo"], collections=b["collections"])
    skymap = butler.get("skyMap", skymap=b["skymap"], collections="skymaps")
    m = iso_cat.iloc[K]                                   # K from section 3
    n = iso["images"].shape[1]
    half = (FACTOR * n) // 2
    p = skymap[int(m.tract)].getWcs().skyToPixel(geom.SpherePoint(float(m.ra), float(m.dec), geom.degrees))
    ix, iy = int(np.floor(p.x + 0.5)), int(np.floor(p.y + 0.5))
    coadd = butler.get("deep_coadd", band=BAND, skymap=b["skymap"], tract=int(m.tract), patch=int(m.patch),
                       parameters={"bbox": Box.factory[iy - half:iy + half + 1, ix - half:ix + half + 1]})
    a = np.asarray(coadd.image.array, float)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.8))
    lo, hi = sym_limits(a, 99.0)
    axes[0].imshow(a, origin="lower", cmap="RdBu_r", vmin=lo, vmax=hi)
    c0 = half - n // 2
    axes[0].add_patch(plt.Rectangle((c0 - 0.5, c0 - 0.5), n, n, fill=False, color="k", lw=1))
    axes[0].set_title(f"coadd {BAND}, {a.shape[0]} px, stamp outlined")
    s = np.asarray(iso["images"][K][..., iso["bands"].index(BAND)], float)
    axes[1].imshow(s, origin="lower", cmap="RdBu_r", vmin=lo, vmax=hi)
    axes[1].set_title("exported stamp (companions filled with sky)")
    for ax in axes:
        ax.grid(False)

## 9. Scratch